In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://www.nature.com/articles/s41467-023-36983-2#Sec34"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41590-021-00922-4/MediaObjects/41590_2021_922_MOESM2_ESM.xlsx"

# don't include in header
table_name = "41467_2023_36983_MOESM4_ESM.xlsx"
table_name = "paper_degs.xlsx" # local name # UPDATE THIS TO BE SPECIFIC FOR HILDRETH

## Read in file

In [ ]:
excel = pd.read_excel(table_name, skiprows = 0)

df = excel

In [ ]:
df.head()

## Perform the filtering

In [9]:
col_pval = "p_val_adj"
col_lfc = "avg_log2FC"
col_gene = "gene"
col_ct = "cluster" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [10]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [ ]:
markers

In [ ]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

In [ ]:
markers[col_ct].value_counts()

## Find Ensembl ID


In [14]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [15]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [16]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [17]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = df["gene"]
df["gene_id"] = df["gene_id"].apply(run)
save = df.to_dict(orient = "records")

In [ ]:
save

## Save evidence

In [19]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 